# 00 - Data: a 51.5M-token slice of FineWeb-Edu

* Source: `HuggingFaceFW/fineweb-edu`, `sample/10BT/000_00000.parquet` (first shard, 2.15 GB)
* Tokenizer: GPT-2 BPE (`tiktoken`), `<|endoftext|>` appended to every document
* First 1.0M tokens -> `data/val.bin` (held out), next 50.5M tokens -> `data/train.bin`
* Every training run reads non-overlapping 512-token windows in the **same fixed shuffled order**,
  so each token is seen exactly once (single epoch) and all runs see identical data.

In [1]:
# --- setup: works on Google Colab (CUDA) and locally (Apple MPS / CUDA / CPU) ---
import os, sys
# MPS: make the allocator raise a clean OOM at the GPU's recommended working-set size
# instead of spilling into swap (must be set before torch is imported).
os.environ.setdefault("PYTORCH_MPS_HIGH_WATERMARK_RATIO", "1.0")
os.environ.setdefault("PYTORCH_MPS_LOW_WATERMARK_RATIO", "0.9")
if "google.colab" in sys.modules and not os.path.exists("revllm.py"):
    !git clone -q -b session13 https://github.com/chapanduk/ERAV5.git /content/ERAV5
    %cd /content/ERAV5/session13
    !pip -q install tiktoken pyarrow
if not os.path.exists("data/train.bin"):
    !python prepare_data.py
import json, torch, revllm
from revllm import ModelConfig, TrainConfig, train, pick_device
DEVICE = pick_device()
print("torch", torch.__version__, "| device", DEVICE,
      "|", torch.cuda.get_device_name() if DEVICE == "cuda" else "")
TRAIN_BIN, VAL_BIN = "data/train.bin", "data/val.bin"
GiB = 2**30

torch 2.15.0 | device mps | 


In [2]:
import numpy as np, tiktoken
tr = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r"); va = np.memmap(VAL_BIN, dtype=np.uint16, mode="r")
print(f"train tokens: {len(tr):,}   val tokens: {len(va):,}")
enc = tiktoken.get_encoding("gpt2")
print("docs in train slice:", int((tr == enc.eot_token).sum()))
print("sample:", repr(enc.decode(tr[:120].tolist())))

train tokens: 50,500,000   val tokens: 1,000,000
docs in train slice: 47946
sample: " artists took great care to make each scene expressive. The mosaics of Bet Alfa are striking in their coloring and style, and are among the finest examples of Jewish art in the Byzantine period.\nKibbutz Bet Alfa was founded at in 1922.\nA scene at one of the children's houses\nMaking up the daily work schedule (August 1946)\nA Man with his cow (August 1930)<|endoftext|>Literary Research Paper\nSimple Ways To Write A Literary Research Paper\nUnlike most other research papers, a literary research paper generally requires an argumentative style of writing rather than a"
